always check the maven repo to see what versions of the iceberg spark runtime are supported
- i generally will go with one version less than the latest

https://mvnrepository.com/artifact/org.apache.iceberg/iceberg-spark-runtime-4.0/versions

Spark is very particular about the version of java you are running
for Spark 4.0+, make sure you are using open jdk 17
- for earlier versions, you will need to set the JAVA_HOME environ var before setting up the spark session

In [ ]:
#checking the version of java
import os
os.system("java -version")

In [ ]:
# if we wanted to run pyspark version 3.3, we would need to downgrade the java home runtime first
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-16.jdk/Contents/Home"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [8]:
# clean up icehouse dir
import shutil
shutil.rmtree("./icehouse", ignore_errors=True)

In [ ]:
spark_version = "4.0" # runs on jdk 17
scala_version = "2.13"
iceberg_version = "1.10.2"

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_date, rand, floor, expr

catalog_name = "iceberg"
namespace = "purch_ord"
warehouse_path = "./icehouse"

spark = SparkSession.builder \
    .appName("local_iceberg_example") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{catalog_name}.type", "hadoop") \
    .config(f"spark.sql.catalog.{catalog_name}.warehouse", warehouse_path) \
    .config("spark.jars.packages", f"org.apache.iceberg:iceberg-spark-runtime-{spark_version}_{scala_version}:{iceberg_version}") \
    .getOrCreate()

In [ ]:
# Iceberg supports namespaces, which are the same as database schemas
spark.sql(f"create namespace if not exists {catalog_name}.{namespace}")

In [ ]:
sql = f"""
CREATE OR REPLACE TABLE {catalog_name}.{namespace}.orders (
    order_id BIGINT,
    order_date DATE,
    customer_id BIGINT,
    total_amount DOUBLE
) USING ICEBERG
"""
spark.sql(sql)

In [ ]:
spark.sql(f"insert into {catalog_name}.{namespace}.orders values (1, current_date(), 1001, 250.75)")

In [ ]:
spark.sql(f"select * from {catalog_name}.{namespace}.orders").show()

In [ ]:
# write to parquet file
spark.sql(f"select * from {catalog_name}.{namespace}.orders").write.mode("overwrite").parquet("./orders_parquet")

In [ ]:
# write to csv file
spark.sql(f"select * from {catalog_name}.{namespace}.orders").write.mode("overwrite").option("header", "true").csv("./orders_csv")